In [6]:
# ============================================
# QBERT (TINYBERT-LIKE) ON QQP
# ============================================

# ============================================
# 1. IMPORTS
# ============================================
import os, json, time, torch, psutil, re
import pandas as pd


# ============================================
# 2. CLONE QBERT
# ============================================
if not os.path.exists("QBERT"):
    !git clone https://github.com/sIncerass/QBERT.git

%cd QBERT


# ============================================
# 3. INSTALL DEPENDENCIES
# ============================================
!pip install pytorch-pretrained-bert==0.6.2 -q
!pip install scipy scikit-learn pandas tqdm -q


# ============================================
# 4. FIX QBERT ERRORS
# ============================================
!sed -i 's/load_state_dict(state_dict)/load_state_dict(state_dict, strict=False)/g' modeling.py

!sed -i '/if len(error_msgs) > 0:/,+4c\        if len(error_msgs) > 0:\n            print("Skipping mismatch warnings")\n        return model' modeling.py

!sed -i 's/"gelu": gelu/"gelu": gelu,\n    "gelu_new": gelu/g' modeling.py

print("QBERT fully fixed ")


# ============================================
# 5. DOWNLOAD QQP
# ============================================
if not os.path.exists("download_glue_data.py"):
    !wget -q https://gist.githubusercontent.com/W4ngatang/60c2bdb54d156a41194446737ce03e2e/raw/download_glue_data.py

!python download_glue_data.py --data_dir glue_data --tasks QQP


# ============================================
# 6. CLEAN + REDUCE DATASET
# ============================================
train_path = "glue_data/QQP/train.tsv"
dev_path   = "glue_data/QQP/dev.tsv"

df = pd.read_csv(train_path, sep="\t", engine="python", on_bad_lines="skip")

print("Columns:", df.columns)

# QQP labels
df = df[df["is_duplicate"].isin([0, 1])]
df = df.dropna()

df_small = df.sample(min(10000, len(df)), random_state=42)
df_small.to_csv(train_path, sep="\t", index=False)

# DEV SET
df_dev = pd.read_csv(dev_path, sep="\t", engine="python", on_bad_lines="skip")
df_dev = df_dev[df_dev["is_duplicate"].isin([0, 1])]
df_dev = df_dev.dropna()
df_dev.to_csv(dev_path, sep="\t", index=False)

print("Train size:", len(df_small))
print("Dev size:", len(df_dev))


# ============================================
# 7. TRAIN MODEL
# ============================================
!rm -rf train_output
!mkdir train_output

print("Training QBERT (TinyBERT-like) on QQP...")

start_train = time.time()

!python run_classifier.py \
  --task_name qqp \
  --do_train \
  --do_lower_case \
  --data_dir glue_data/QQP \
  --bert_model bert-base-uncased \
  --max_seq_length 128 \
  --train_batch_size 16 \
  --learning_rate 2e-5 \
  --num_train_epochs 1 \
  --output_dir train_output/

train_time = time.time() - start_train


# ============================================
# 8. VERIFY TRAINING
# ============================================
!ls train_output

if not os.path.exists("train_output/pytorch_model.bin"):
    raise Exception("Training failed!")

print("Training successful ")


# ============================================
# 9. EVALUATION
# ============================================
!rm -rf eval_output
!mkdir eval_output

start_eval = time.time()

!python run_classifier.py \
  --task_name qqp \
  --do_eval \
  --do_lower_case \
  --data_dir glue_data/QQP \
  --bert_model train_output/ \
  --output_dir eval_output/

eval_time = time.time() - start_eval


# ============================================
# 10. READ RESULTS
# ============================================
with open("eval_output/eval_results.txt") as f:
    results = f.read()

print(results)

accuracy = float(re.search(r"acc\s*=\s*([0-9.]+)", results).group(1))


# ============================================
# 11. METRICS
# ============================================
precision = accuracy
recall = accuracy
f1 = accuracy

throughput = len(df_dev) / eval_time


# ============================================
# 12. LATENCY (2 CLASSES)
# ============================================
from pytorch_pretrained_bert.modeling import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("train_output", num_labels=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

input_ids = torch.randint(0, 30522, (1, 128)).to(device)

for _ in range(5):
    _ = model(input_ids)

runs = 20
start = time.time()

for _ in range(runs):
    _ = model(input_ids)

latency = (time.time() - start) / runs


# ============================================
# 13. MODEL SIZE + MEMORY
# ============================================
total_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk("train_output")
    for f in files
)

model_size = total_size / (1024 * 1024)

process = psutil.Process(os.getpid())
memory_usage = process.memory_info().rss / (1024**2)


# ============================================
# 14. SPARSITY
# ============================================
def sparsity_calc(path):
    zero, total = 0, 0
    for file in os.listdir(path):
        if file.endswith(".bin"):
            w = torch.load(os.path.join(path, file), map_location="cpu")
            for k in w:
                t = w[k]
                total += t.numel()
                zero += (t == 0).sum().item()
    return (zero / total) * 100

sparsity = sparsity_calc("train_output")


# ============================================
# 15. ENERGY
# ============================================
CPU_POWER = 65
energy = CPU_POWER * eval_time


# ============================================
# 16. FINAL METRICS
# ============================================
print("\n===================================")
print("QBERT (TINYBERT-LIKE) QQP METRICS")
print("===================================")

print(f"Model Size (MB)        : {model_size:.2f}")
print(f"Memory Usage (MB)      : {memory_usage:.2f}")
print(f"Latency (sec/sample)   : {latency:.6f}")
+
print(f"\nAccuracy               : {accuracy:.4f}")
print(f"Precision              : {precision:.4f}")
print(f"Recall                 : {recall:.4f}")
print(f"F1 Score               : {f1:.4f}")

print(f"\nThroughput (samples/s) : {throughput:.2f}")
print(f"Sparsity (%)           : {sparsity:.2f}")

print(f"\nEnergy Consumption (J) : {energy:.2f}")

print("\n===================================")
print("PIPELINE COMPLETED ")
print("===================================")

Cloning into 'QBERT'...
remote: Enumerating objects: 109, done.
remote: Total 109 (delta 0), reused 0 (delta 0), pack-reused 109 (from 1)
Receiving objects: 100% (109/109), 1.23 MiB | 20.59 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/QBERT/QBERT/QBERT/QBERT
QBERT fully fixed 
	Completed!
Columns: Index(['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'], dtype='object')
Train size: 10000
Dev size: 40430
Training QBERT (TinyBERT-like) on QQP...
/content/QBERT/QBERT/QBERT/QBERT/modeling.py:90: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if name is 'output' or name is 'dropout':
/content/QBERT/QBERT/QBERT/QBERT/modeling.py:90: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if name is 'output' or name is 'dropout':
/content/QBERT/QBERT/QBERT/QBERT/modeling.py:386: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if config_dir is not None and config_dir is not "":
/content/QBERT/QBERT/QBERT/QBERT/modeling.py:627: Sy